<a href="https://colab.research.google.com/github/KatyaKom/RSS26/blob/main/Part2/vancomycert_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Property-Driven Training for Drug Dosing

**Summer School in Robotics — Student Notebook**

In this notebook you will:
1. Simulate a cohort of patients receiving a drug and generate training data
2. Train a baseline neural network using mean squared error only
3. Write formal safety properties in Vehicle and compile them into loss functions
4. Train a network with GradNorm-balanced constraint losses and see it pass formal verification

---
**Run time:** ~15–20 minutes on a Colab CPU  
**No GPU needed** — the models are small

## 0. Setup

Clone the project repo and install the non-standard dependencies.  
TensorFlow and scikit-learn are already available in Colab.

This cell ensures all necessary Python packages are installed, the project repository is cloned from GitHub, and the custom `drug_verification` package is installed in editable mode. It also sets up the Python path for correct module imports.

In [ ]:
import os
import sys
import shutil as _sh

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install dependencies not pre-installed on Colab
    !pip install vehicle-lang idx2numpy tf2onnx -q

    # Clone the project repository
    !rm -rf /content/RSS26
    !cd /content && git clone https://github.com/KatyaKom/RSS26.git || echo "Git clone failed. Please check your internet connection or try again."

    REPO_PATH = '/content/RSS26/Part2'
    os.chdir(REPO_PATH)

    SRC_PATH = os.path.join(REPO_PATH, 'src')
    if SRC_PATH not in sys.path:
        sys.path.insert(0, SRC_PATH)
    !pip install -e .

    # Install Marabou: wheel first, source build only as fallback
    try:
        import maraboupy
        print("maraboupy already importable.")
    except ImportError:
        print("Installing maraboupy wheel...")
        !pip install -q maraboupy==2.0.0
        try:
            import maraboupy
            print("maraboupy wheel installed successfully.")
        except ImportError:
            print("Wheel install failed — falling back to source build (~20 min)...")
            import glob
            _repo  = os.path.join(os.getcwd(), 'Marabou-src')
            _build = os.path.join(_repo, 'build')
            !apt-get install -qq cmake
            if not os.path.exists(_repo):
                !git clone -q --depth 1 --branch v2.0.0 https://github.com/NeuralNetworkVerification/Marabou.git {_repo}
            !cmake -S {_repo} -B {_build} -DCMAKE_BUILD_TYPE=Release -DBUILD_PYTHON=OFF -DRUN_UNIT_TEST=OFF -DRUN_REGRESS_TEST=OFF -DRUN_SYSTEM_TEST=OFF
            !cmake --build {_build} --target Marabou -j$(nproc) 2>&1 | tail -5
            _bins = [p for p in glob.glob(os.path.join(_build, '**', 'Marabou'), recursive=True)
                     if os.path.isfile(p) and os.access(p, os.X_OK)]
            if not _bins:
                raise RuntimeError(f'Marabou binary not found after build. Searched: {_build}')
            import shutil
            shutil.copy2(_bins[0], '/usr/local/bin/Marabou')
            os.chmod('/usr/local/bin/Marabou', 0o755)
            print('Marabou executable installed to /usr/local/bin/Marabou')
else:
    # Running locally — dependencies should already be installed in the venv.
    # Ensure the src directory is on the path.
    REPO_PATH = os.path.dirname(os.path.abspath('pk.vcl'))
    SRC_PATH = os.path.join(REPO_PATH, 'src')
    if SRC_PATH not in sys.path:
        sys.path.insert(0, SRC_PATH)
    print("Running locally. Skipping install steps.")

print("Marabou on PATH:", _sh.which("Marabou"))
print('Setup complete.')

After the setup is complete, this cell imports all required libraries and modules for the notebook's operation and sets the random seeds for reproducibility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU for reproducibility
import tensorflow as tf

from drug_verification.simulation import simulate_cohort
from drug_verification.types import SimulationConfig
from drug_verification.training import (
    build_model, prepare_data, train_model,
    train_model_with_constraint, evaluate_model, update_vcl_scaler, export_onnx
)
from drug_verification.vehicle_loss import load_drug_verification_constraints
from drug_verification import constants as C

tf.random.set_seed(42)
np.random.seed(42)

---
## 1. The Problem: Drug Dosing Under Uncertainty

We want to train a neural network to recommend drug doses for patients receiving an antimicrobial.
The network takes five inputs at each timestep:

| Input | Meaning | Units |
|-------|---------|-------|
| `conc` | Current plasma drug concentration | µg/mL |
| `temp` | Body temperature | °C |
| `wbc` | White blood cell count | ×10³/µL |
| `age` | Patient age | years |
| `weight` | Patient weight | kg |

And outputs a single recommended dose in **mg**.

The key safety constraint: plasma concentration must not exceed **C_safe = 30 µg/mL** (toxic threshold).

### 1.1 Generating training data

We simulate 50 patients over 48 timesteps using a two-compartment PK/PD model.  
Drug concentration evolves as:

$$C(t) = \frac{D \cdot K_a}{V_d (K_a - K_e)} \left( e^{-K_e t} - e^{-K_a t} \right)$$

Each patient has individual PK parameters derived from their age and weight.  
A feedback controller generates the dose label for each state.

In [ ]:
cfg = SimulationConfig(num_patients=50, timesteps=48, seed=42)
X, y = simulate_cohort(cfg, seed=42)

print(f'Training data shape: X={X.shape}, y={y.shape}')
print(f'Features: concentration, temperature, WBC, age, weight')
print(f'Dose range: {y.min():.1f} – {y.max():.1f} mg')
print(f'Mean dose: {y.mean():.1f} mg')

In [ ]:
# Plot the first 3 patients' trajectories
n_steps = cfg.timesteps - 1  # 47 per patient

fig, axes = plt.subplots(3, 3, figsize=(13, 8))
feature_names = ['Concentration (µg/mL)', 'Temperature (°C)', 'WBC (×10³/µL)']
colours = ['#2196F3', '#F44336', '#4CAF50']

for patient_idx in range(3):
    start = patient_idx * n_steps
    end = start + n_steps
    X_p = X[start:end]
    y_p = y[start:end]
    t = np.arange(n_steps)

    for feat_idx in range(3):
        ax = axes[patient_idx, feat_idx]
        ax.plot(t, X_p[:, feat_idx], color=colours[feat_idx], lw=2)
        if feat_idx == 0:
            ax.axhline(30, color='red', ls='--', lw=1.5, label='C_safe = 30')
            ax.set_ylabel(f'Patient {patient_idx+1}', fontsize=11)
        if patient_idx == 0:
            ax.set_title(feature_names[feat_idx])
        ax.set_xlabel('Timestep')

plt.tight_layout()
plt.suptitle('Patient State Trajectories (first 3 patients)', y=1.02, fontsize=13)
plt.show()

In [ ]:
# Distribution of doses
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(y, bins=50, color='#2196F3', edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Dose (mg)')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of recommended doses')

ax2.scatter(X[:, 1], y, alpha=0.3, s=10, c='#F44336')
ax2.set_xlabel('Temperature (°C)')
ax2.set_ylabel('Dose (mg)')
ax2.set_title('Dose vs Temperature')

plt.tight_layout()
plt.show()

---
## 2. Formalising Safety in Vehicle

**Accuracy is not safety.**  
A network that achieves low MSE on training data can still recommend dangerous doses for specific patient states.

We want something stronger: a *proof* that for **every** patient state in the physiological range, the network will not recommend a dose that pushes concentration above C_safe.

Vehicle lets us write this as a formal `@property` and then either:
- **Compile it into a differentiable loss** for property-driven training, or  
- Pass it to Marabou for **formal verification** (exhaustive search for counterexamples)

> **Key insight:** the *same* `.vcl` specification file drives both training and verification. The `vehicle` compiler translates `@property` declarations directly into TensorFlow loss functions — you are not writing a separate proxy loss by hand; the loss *is* the formal spec.

### The Vehicle specification (`pk.vcl`)

In [ ]:
# Read and display the Vehicle specification
with open('pk.vcl', 'r') as f:
    spec = f.read()
print(spec)

### Understanding the key properties

**`safe`** — For every patient state in the physiological range:
```vehicle
safe = forall x . safeInput x => safeOutput x
```
The `safeOutput` predicate applies the peak-concentration formula to the network's output
and checks the result stays below C_safe.

**`nonNeg`** — Doses are never negative:
```vehicle
nonNeg = forall x . safeInput x => 0 <= (normpk x) ! 0
```

> **Key point for roboticists:** `forall x` quantifies over the *entire* input space defined by `safeInput`. This is fundamentally different from evaluating on a test set. A test set is finite; the input box is continuous and infinite.


In [ ]:
# What does safeInput cover?
print('safe input region:')
print(f'  concentration: [0, {C.DEFAULT_SPEC_PARAMS["C_safe"]:.1f}] µg/mL  (full physiological range up to C_safe=30)')
print(f'  temperature:   [36.5, 40.0] °C')
print(f'  WBC:           [7.5, 20.0] ×10³/µL')
print(f'  age:           [18, 89] years')
print(f'  weight:        [50, 100] kg')
print()
print('This is a continuous 5D box — infinitely many points.')
print('Marabou searches all of them for a counterexample.')


---
## 3. Baseline: Training Without Safety Constraints

First, train a standard neural network using only MSE.  
We will then check it against the Vehicle property — it will fail.

The cell below also compiles `pk.vcl` into a callable TensorFlow loss via `load_drug_verification_constraints`. This is the same spec used later for Marabou verification — there is no separate loss function written by hand.

In [ ]:
# Prepare data — StandardScaler on X, y stays in raw mg for the baseline
X_train, X_test, y_train, y_test, scaler, y_scaler = prepare_data(X, y, seed=42)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')
print(f'X scaler mean:    {scaler.mean_.round(2)}')
print(f'X scaler std:     {scaler.scale_.round(2)}')
print(f'y scaler mean:    {y_scaler.mean_[0]:.2f} mg  (dose centre)')
print(f'y scaler std:     {y_scaler.scale_[0]:.2f} mg  (dose scale)')

In [ ]:
# Build a standard feedforward network: Input(5) → Dense(128) → Dense(64) → Dense(1)
baseline_model = build_model(input_size=5)
baseline_model.summary()

In [ ]:
# Train on MSE only (standard Keras fit, no Vehicle loss)
baseline_history = train_model(
    baseline_model, X_train, y_train, X_test, y_test,
    epochs=30, batch_size=32
)

baseline_metrics = evaluate_model(baseline_model, X_test, y_test, y_scaler=y_scaler)
print(f"\nBaseline model:")
print(f"  MAE:       {baseline_metrics['mae']:.2f} mg")
print(f"  Max error: {baseline_metrics['max_absolute_error']:.2f} mg")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(9, 4))
hist = baseline_history.history
ax.plot(hist['loss'], label='Train loss (MSE, normalised)', color='#2196F3')
ax.plot(hist['val_loss'], label='Val loss', color='#2196F3', ls='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (normalised dose space)')
ax.set_title('Baseline Training — MSE only')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check the baseline model against the Vehicle safe constraint
# We use the Vehicle loss as a post-hoc diagnostic — a non-zero value means violations exist
update_vcl_scaler(scaler, y_scaler=y_scaler, spec_path='pk.vcl')

constraint_fns = load_drug_verification_constraints(
    spec_path='pk.vcl',
    properties=['safe'],
    mean=scaler.mean_,
    std_dev=scaler.scale_,
    parameters=C.DEFAULT_SPEC_PARAMS,
)

y_mean = float(y_scaler.mean_[0])
y_std  = float(y_scaler.scale_[0])

def make_network_fn(model, y_mean, y_std, training=False):
    # training=False is correct for diagnostic checks here.
    # Inside train_model_with_constraint the equivalent wrapper uses training=True
    # so that dropout/batch-norm layers behave correctly during the training forward pass.
    def network_fn(x):
        out = tf.reshape(model(tf.reshape(x, [1, -1]), training=training), [-1])
        return out * y_std + y_mean
    return network_fn

baseline_net_fn = make_network_fn(baseline_model, y_mean, y_std)

safe_loss = float(constraint_fns['safe'](pk=baseline_net_fn).numpy())

print(f'\nBaseline model — Vehicle constraint loss (> 0 means violations):')
print(f'  safe loss: {safe_loss:.4f}')
print()
if safe_loss > 0:
    print('The baseline model VIOLATES safety properties.')
    print('MSE being low does not guarantee formal safety.')
else:
    print('Loss is 0 — model may already satisfy properties (unusual for a baseline).')


---
## 4. Property-Driven Training with GradNorm

We now add the Vehicle constraint losses directly to the training objective.  
The challenge: these three losses have very different magnitudes.

### 4.1 The Loss Scale Problem

| Loss | Typical value at start of constraint training |
|------|-----------------------------------------------|
| Task (MSE, normalised) | ~1.0 |
| `safe` | ~46 |

A fixed weighting (e.g. 50% task / 50% constraint) can cause gradient imbalance and collapse the task loss.

**GradNorm** (Chen et al., ICML 2018) adapts weights automatically so that both  
losses train at the same *rate* relative to their starting values.

In [ ]:
# Demonstrate the loss scale problem before normalisation
fresh_model = build_model(input_size=5)
net_fn_raw = make_network_fn(fresh_model, y_mean, y_std)

# Task loss on a batch of training data
X_batch = tf.constant(X_train[:32], dtype=tf.float32)
y_batch = tf.constant(y_train[:32], dtype=tf.float32)
preds = fresh_model(X_batch, training=False)
task_loss = float(tf.reduce_mean(tf.square(preds - y_batch)).numpy())

# Constraint loss
safe_l = float(constraint_fns['safe'](pk=net_fn_raw).numpy())

print('Loss magnitudes from a fresh (randomly initialised) model:')
print(f'  Task (MSE, normalised): {task_loss:.4f}')
print(f'  safe:                   {safe_l:.4f}')
print()
print('GradNorm will normalise each by its initial value so all start at 1.0,')
print('then adapt weights based on relative training rates.')


### 4.2 Two-Phase Training

Starting GradNorm constraint balancing from epoch 1 on a randomly initialised network  
overwhelms the task signal — the model never learns clinically reasonable doses.

**Phase 1 (epochs 1–N):** task loss only — learn to predict reasonable doses first.  
**Phase 2 (epochs N+1–end):** GradNorm balances task + safe.

At the phase boundary, the Adam optimiser is reset so its accumulated momentum  
estimates (calibrated to small task gradients) don't cause overshooting when the  
much larger constraint gradients first appear.

In [ ]:
# Build a fresh model for PDT training
pdt_model = build_model(input_size=5)

pdt_history = train_model_with_constraint(
    pdt_model,
    X_train, y_train, X_test, y_test,
    constraint_fn=constraint_fns['safe'],
    y_mean=y_mean,
    y_std=y_std,
    alpha=0.5,           # GradNorm (exercise 3)
    epochs=100,          # Epochs (exercise 1)
    batch_size=32,
    phase_switch=30,     # Phase Switch (exercise 2)
    verbose=True,
)


In [ ]:
# Plot the training curves
epochs_range = range(1, len(pdt_history['task_loss']) + 1)
phase_switch_epoch = 5

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: losses — linear scale because constraint losses are 0 during phase 1
# (log(0) is undefined; matplotlib would drop those points and emit a warning)
ax = axes[0]
ax.plot(epochs_range, pdt_history['task_loss'],       label='Task (MSE)', color='#2196F3', lw=2)
ax.plot(epochs_range, pdt_history['constraint_loss'], label='safe',       color='#F44336', lw=2)
ax.axvline(phase_switch_epoch, color='gray', ls=':', lw=1.5, label=f'Phase switch (epoch {phase_switch_epoch})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Per-task losses')
ax.legend()

# Right: adaptive weights
ax2 = axes[1]
ax2.plot(epochs_range, pdt_history['task_weight'],       label='Task weight',  color='#2196F3', lw=2)
ax2.plot(epochs_range, pdt_history['constraint_weight'], label='safe weight',  color='#F44336', lw=2)
ax2.axvline(phase_switch_epoch, color='gray', ls=':', lw=1.5, label='Phase switch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('GradNorm weight')
ax2.set_title('Adaptive GradNorm weights')
ax2.legend()

plt.suptitle('Property-Driven Training', fontsize=13)
plt.tight_layout()
plt.show()


### 4.3 Understanding the weight evolution

Watch the weight plot above:
- During phase 1 (epochs 1–5), only the task weight is active (logged as 1.0); the constraint weight is 0 — GradNorm is not yet running
- At the phase switch, GradNorm initialises and normalises each loss by its initial value so both start at ~1.0
- Over time, the task weight typically stabilises lower as the model satisfies the `safe` constraint


In [ ]:
# Evaluate PDT model and compare with baseline
pdt_metrics = evaluate_model(pdt_model, X_test, y_test, y_scaler=y_scaler)

print('─' * 50)
print(f'{"Metric":<25} {"Baseline":>10} {"PDT":>10}')
print('─' * 50)
print(f'{"MAE (mg)":<25} {baseline_metrics["mae"]:>10.2f} {pdt_metrics["mae"]:>10.2f}')
print(f'{"Max error (mg)":<25} {baseline_metrics["max_absolute_error"]:>10.2f} {pdt_metrics["max_absolute_error"]:>10.2f}')
print('─' * 50)

# Check Vehicle losses on PDT model
pdt_net_fn = make_network_fn(pdt_model, y_mean, y_std)
pdt_safe = float(constraint_fns['safe'](pk=pdt_net_fn).numpy())

print()
print(f'{"":25} {"Baseline":>10} {"PDT":>10}')
print('─' * 50)
print(f'{"safe loss":<25} {safe_loss:>10.4f} {pdt_safe:>10.4f}')
print('─' * 50)


In [ ]:
# Prediction scatter: baseline vs PDT
y_test_raw = y_scaler.inverse_transform(np.array(y_test).reshape(-1, 1)).flatten()

baseline_preds_raw = y_scaler.inverse_transform(
    baseline_model.predict(X_test, verbose=0).reshape(-1, 1)
).flatten()
pdt_preds_raw = y_scaler.inverse_transform(
    pdt_model.predict(X_test, verbose=0).reshape(-1, 1)
).flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
lims = [y_test_raw.min() - 20, y_test_raw.max() + 20]

for ax, preds, title in zip(axes,
                             [baseline_preds_raw, pdt_preds_raw],
                             ['Baseline (MSE only)', 'PDT + GradNorm']):
    ax.scatter(y_test_raw, preds, alpha=0.3, s=10, color='#2196F3')
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel('True dose (mg)')
    ax.set_ylabel('Predicted dose (mg)')
    ax.set_title(title)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 5. Formal Verification

After training, we export the model to ONNX and ask Marabou to exhaustively search  
for a counterexample in the `safe` input region.

### Export

The Keras model uses a linear output activation.  
`export_onnx()` appends `Relu + Add(0.0001)` to the ONNX graph to guarantee  
non-negative outputs and satisfy the `nonNeg` property by construction.

In [ ]:
import os
os.makedirs('models', exist_ok=True)
export_onnx(pdt_model, out_path='models/pk_tutorial.onnx')
print('Model exported to models/pk_tutorial.onnx')

In [ ]:
# Verify with Vehicle + Marabou
import subprocess

params = C.DEFAULT_SPEC_PARAMS
param_args = []
for k, v in params.items():
    param_args += ['--parameter', f'{k}:{v}']

cmd = [
    'vehicle', 'verify',
    '--specification', 'pk.vcl',
    '--network', f'pk:models/pk_tutorial.onnx',
    '--verifier', 'Marabou',
] + param_args

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)


---
## 6. Clinical Relevance Check

A pathologically conservative model could pass verification by always predicting  
zero (or near-zero) doses. We must confirm the model is actually clinically responsive.

In [ ]:
# Evaluate on all patients
X_all_scaled = scaler.transform(X)
all_preds_norm = pdt_model.predict(X_all_scaled, verbose=0)
all_preds = y_scaler.inverse_transform(all_preds_norm.reshape(-1, 1)).flatten()

errors = all_preds - y
corr = float(np.corrcoef(y, all_preds)[0, 1])
pred_std_ratio = float(np.std(all_preds) / np.std(y))

print('Clinical evaluation (all 2350 samples):')
print(f'  MAE:              {np.mean(np.abs(errors)):.1f} mg')
print(f'  95th pct error:   {np.percentile(np.abs(errors), 95):.1f} mg')
print(f'  Max absolute err: {np.max(np.abs(errors)):.1f} mg')
print(f'  Mean bias:        {np.mean(errors):+.1f} mg')
print(f'  Corr(true, pred): {corr:.3f}  (1.0 = perfect)')
print(f'  Pred std / True std: {pred_std_ratio:.3f}  (1.0 = fully responsive)')
print()

# Check model responsiveness: low-temp vs high-temp doses
low_temp_mask  = X[:, 1] < 37.5
high_temp_mask = X[:, 1] > 39.0
print(f'  Avg dose — low temp  (<37.5°C): {all_preds[low_temp_mask].mean():.1f} mg')
print(f'  Avg dose — high temp (>39.0°C): {all_preds[high_temp_mask].mean():.1f} mg')
ratio = all_preds[high_temp_mask].mean() / (all_preds[low_temp_mask].mean() + 1e-6)
print(f'  Ratio: {ratio:.1f}×  (> 1 means model prescribes more for sicker patients)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Error distribution
axes[0].hist(errors, bins=50, color='#2196F3', edgecolor='white', linewidth=0.5)
axes[0].axvline(0, color='red', ls='--', lw=1.5)
axes[0].set_xlabel('Prediction error (mg)')
axes[0].set_ylabel('Count')
axes[0].set_title('Error distribution')

# Pred vs true
axes[1].scatter(y, all_preds, alpha=0.2, s=8, color='#4CAF50')
lims = [y.min() - 20, y.max() + 20]
axes[1].plot(lims, lims, 'r--', lw=1.5)
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel('True dose (mg)')
axes[1].set_ylabel('Predicted dose (mg)')
axes[1].set_title(f'Predictions (r={corr:.3f})')
axes[1].set_aspect('equal')

# Dose vs temperature
axes[2].scatter(X[:, 1], all_preds, alpha=0.2, s=8, color='#F44336')
axes[2].set_xlabel('Temperature (°C)')
axes[2].set_ylabel('Predicted dose (mg)')
axes[2].set_title('Clinical responsiveness')

plt.tight_layout()
plt.suptitle('PDT Model — Clinical Evaluation', fontsize=13, y=1.02)
plt.show()

---
## Summary

| Step | What we did |
|------|-------------|
| Simulation | Generated PK/PD patient data (50 patients, 47 timesteps each) |
| Spec | Wrote `safe` as a `forall` property in Vehicle |
| Baseline | Showed MSE training produces good accuracy but violates safety properties |
| PDT | Compiled `safe` → differentiable loss; GradNorm balanced task + constraint |
| Verification | Marabou exhaustively searched the input space — no counterexample found |
| Clinical | Confirmed the model is responsive, not pathologically conservative |

### Key engineering decisions

| Problem | Fix |
|---------|-----|
| Loss scale mismatch (MSE vs constraint) | Normalise y targets with `StandardScaler` |
| `safe` branch on symbolic parameters | Inline PK params in generated temp `.vcl` |
| GradNorm weight stuck at floor | Normalise each loss by its initial value |
| `tape.gradient` returns zeros | Compute per-loss gradients individually, combine manually |
| Adam overshoot at phase switch | Reset optimiser with fresh Adam at `phase_switch` |

---
## Exercises

1. **Does it verify with less `epochs`*** Compare across different sizes of training, is the model more conservative, is this clincally relevant?

2. **Change `phase_switch`** to 0 (constraints from epoch 1) and observe what happens to the loss curves. Why does the task loss collapse?

3. **Change `alpha`** (GradNorm exponent) from 0.5 to 2.0. Does the training become more or less stable? What do the weights look like?

4. **Adjust the `pk.vcl`**, notice something about CONC and C_Safe? How does adjusting the ***`safe`*** property impact its training and verification?

5. **The baseline model sometimes passes verification without constraint training.** Why might a model trained only on MSE still happen to satisfy the `safe` property? And why does longer training or a different random seed make failure more likely?